## Overview

Before we start with processing, we need to cut individual videos corresponding to item production from the session videos. We will use existing ELAN annotations to do this, or a PsychoPy file.

In [ ]:
#| code-fold: true
#| code-summary: Code to prepare the environment

import os
import glob
from pympi.Elan import Eaf
import cv2
import numpy as np
import pandas as pd

curfolder = os.getcwd()

## Cut videos to trial-sized clips

### Cut learners

In [ ]:
videofolder = os.path.join(curfolder, 'Videodata/IP/')
elanfolder = os.path.join(curfolder, 'Videodata/IP/IP EAF/')
outputfolder = os.path.join(curfolder, 'Videodata/IP/IP_trials/')

videofiles = glob.glob(videofolder + '*.mp4')
elanfiles = glob.glob(elanfolder + '*.eaf')

tier = 'Sign/gesture'

In [ ]:
# make a dictionary of the lags for each video_name
vidnames = []
for video in videofiles:
    vidname = video.split('\\')[-1].split('.')[0]
    # keep only first two elements
    vidname = '_'.join(vidname.split('_')[:2])
    vidnames.append(vidname)

# now make a dictionary of the lags for each vid_name, put there 0 for now
vidlags = {vidname: 0 for vidname in vidnames}

In [ ]:
# add individual lags (based on manual inspection)
vidlags['IP06_02'] = 250
vidlags['IP08_02'] = 1050
vidlags['IP10_02'] = 550
vidlags['IP16_02'] = -50

In [ ]:
for video in videofiles:

    # Get the name of the video file
    video_name = os.path.basename(video).split('.')[0]
    # Keep only first two elements of the name
    video_name = video_name.split('_')[0] + '_' + video_name.split('_')[1]
    # Get the corresponding ELAN file
    elanfile = [f for f in elanfiles if video_name in f][0]
    print(video_name)
    print(elanfile)
    # Load the ELAN file
    eaf = Eaf(elanfile)
    # Get the annotations from the tier
    annotations = eaf.get_annotation_data_for_tier(tier)

    # Get the video
    cap = cv2.VideoCapture(video)
    # Get the frame rate
    fps = cap.get(cv2.CAP_PROP_FPS)
    # Get the length of the video by time
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps)
    print('length of the video: ' + str(length))
    # Number of frames
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print('number of frames: ' + str(frames))
    
    # Process the annotations
    for anno in annotations:
        print('annotation: ' + anno[2])
        start_time = anno[0] / 1000  # Convert ms to seconds
        end_time = anno[1] / 1000  # Convert ms to seconds
        annotation_value = anno[2]

        # Calculate start and end frames
        start_frame = int(start_time * fps)
        end_frame = int(end_time * fps)

        frames = range(start_frame, end_frame)

        # If annotation value contains Ä, Ö, Ü, ß, replace it with AE, OE, UE, SS (capital)
        annotation_value = annotation_value.replace('Ä', 'AE')
        annotation_value = annotation_value.replace('Ö', 'OE')
        annotation_value = annotation_value.replace('Ü', 'UE')
        annotation_value = annotation_value.replace('ß', 'SS')

        # Set output filename
        outputfilename = os.path.join(outputfolder, video_name + '_' + annotation_value + '.mp4')

        # Check if the video already exists
        if os.path.exists(outputfilename):
            print('video already exists: ' + outputfilename)
            continue

        # Define videowriter
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        out = cv2.VideoWriter(outputfilename, fourcc, fps, (width, height))

        for fra in frames:
            # Add 100 frames because we are not sure if the frames are loaded correctly
            # Get lag from the dictionary
            fra_lag = fra + vidlags[video_name]
            cap.set(cv2.CAP_PROP_POS_FRAMES, fra_lag)
            ret, frame = cap.read()
            if ret:
                # display th loading image with im.show() to see if the frames are loaded correctly
                #cv2.imshow('frame',frame)
                #cv2.waitKey(1000)
                out.write(frame)
                
            if not ret:
                print('a frame was dropped: ' + str(fra))

        out.release()

    cap.release()
    cv2.destroyAllWindows()

print('done')

### Cut gesturers

In [ ]:
videofolder = os.path.join(curfolder, 'data/Gestures/')
outputfolder = os.path.join(videofolder, 'Gestures_trials/')
csvfolder = os.path.join(videofolder, 'GP_files/')

videofiles = glob.glob(videofolder + '*.mp4')
csvfiles = glob.glob(csvfolder + '*.csv')

# read in excel file Offset LP
offsets = pd.read_excel(os.path.join(csvfolder, 'Offset GP.xlsx'))

In [ ]:
## 01 offset by 150
## 15 offset by 50
## 16 offset by 100

for video in videofiles:

    # Get the name of the video file
    video_name = os.path.basename(video).split('.')[0]
    # Keep only first two elements of the name
    video_name = video_name.split('_')[0].split('-')[0]
    # Get the corresponding csv file
    csvfile = [file for file in csvfiles if video_name in file][0]
    # Load the psychopy file
    df = pd.read_csv(csvfile)

    # Get item_DE, SignStart, SignEnd
    subdf = df[['text_de', 'text_Prompt.started']]
    # Discard rows where item_DE is NaN
    subdf = subdf.dropna(subset=['text_de'])

    # Get the video
    cap = cv2.VideoCapture(video)
    # Get the frame rate
    fps = cap.get(cv2.CAP_PROP_FPS)
    # Get the length of the video by time
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps)
    print('length of the video: ' + str(length))
    # Number of frames
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print('number of frames: ' + str(frames))
    
    # Loop through the subdf
    for index, row in subdf.iterrows():
        # print what item_DE is being processed
        #print(row['text_de'])
        # Get the start and end, convert to seconds (it's in mm:ss:ms)
        start_time = row['text_Prompt.started']
        # Get the row where first column has value video_name
        offrow = offsets[offsets.iloc[:,0] == video_name]
        # Get the offset from the coumn Offset in this row
        offset = offrow['Offset']
        #print('offset: ' + str(offset))
        start_time = start_time + offset
        # End is start + 4 seconds
        end_time = start_time + 4
        annotation_value = row['text_de']
        print(annotation_value)

        # Calculate start and end frames
        start_frame = int(start_time * fps)
        end_frame = int(end_time * fps)
        frames = range(start_frame, end_frame)

        # If annotation value contains Ä, Ö, Ü, ß, replace it with AE, OE, UE, SS (capital)
        annotation_value = annotation_value.replace('Ä', 'AE')
        annotation_value = annotation_value.replace('Ö', 'OE')
        annotation_value = annotation_value.replace('Ü', 'UE')
        annotation_value = annotation_value.replace('ß', 'SS')

        # Set output filename
        outputfilename = os.path.join(outputfolder, video_name + '_' + annotation_value + '.mp4')

        # Check if the video already exists
        if os.path.exists(outputfilename):
            print('video already exists: ' + outputfilename)
            continue

        # Define videowriter
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        out = cv2.VideoWriter(outputfilename, fourcc, fps, (width, height))

        for fra in frames:
            # Add 100 frames because we are not sure if the frames are loaded correctly
            # Get lag from the dictionary
            fra_lag = fra #+ vidlags[video_name]
            cap.set(cv2.CAP_PROP_POS_FRAMES, fra_lag)
            ret, frame = cap.read()
            if ret:
                # display th loading image with im.show() to see if the frames are loaded correctly
                #cv2.imshow('frame',frame)
                #cv2.waitKey(1000)
                out.write(frame)
                
            if not ret:
                print('a frame was dropped: ' + str(fra))

        out.release()

    cap.release()
    cv2.destroyAllWindows()

print('done')